In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Data Preprocessing

In [3]:
# 1. LOAD THE DATA
# Note: You accidentally loaded Data_Klaim twice in your prompt. I fixed the second one.
claims_df = pd.read_csv('data/Data_Klaim.csv')
polis_df = pd.read_csv('data/Data_Polis.csv')

In [4]:
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
0,C-0001-M,POL-0176,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,2024-07-08,2024-05-27,2024-05-27,28093653.0,6.143948e+06,Singapore
1,C-0002-M,POL-3288,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-08-06,2024-07-15,2024-07-15,80987278.0,8.230952e+07,Malaysia
2,C-0003-M,POL-1786,R,OP,C18.9,"MALIGNANT NEOPLASM, COLON, UNSPECIFIED",PAID,2024-10-17,2024-05-16,2024-05-16,183047130.0,1.928599e+08,Singapore
3,C-0004-M,POL-1786,R,OP,C34,MALIGNANT NEOPLASM OF BRONCHUS AND LUNG,PAID,2024-09-03,2024-07-18,2024-07-18,191424386.0,1.914244e+08,Singapore
4,C-0005-M,POL-2778,R,OP,C50,MALIGNANT NEOPLASM OF BREAST,PAID,NaN,2024-06-06,2024-06-06,138936357.0,1.389364e+08,Singapore


In [5]:
polis_df.head()

,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,20140603,JAKARTA
1,POL-0002,M-003,M,19710730,20140603,JAKARTA
2,POL-0003,M-001,M,19790821,20160808,JAKARTA
3,POL-0004,M-003,M,20140724,20160811,JAKARTA
4,POL-0005,M-001,F,19810114,20150828,JAKARTA


In [6]:
# 2. CALCULATE EXPOSURE FROM POLIS TABLE (CRITICAL FOR FREQUENCY)
# Convert Tanggal Efektif Polis (format YYYYMMDD) to datetime
polis_df['Tanggal Efektif Polis'] = pd.to_datetime(polis_df['Tanggal Efektif Polis'], format='%Y%m%d', errors='coerce')
polis_df.head()

,Nomor Polis,Plan Code,Gender,Tanggal Lahir,Tanggal Efektif Polis,Domisili
0,POL-0001,M-003,M,19640811,2014-06-03,JAKARTA
1,POL-0002,M-003,M,19710730,2014-06-03,JAKARTA
2,POL-0003,M-001,M,19790821,2016-08-08,JAKARTA
3,POL-0004,M-003,M,20140724,2016-08-11,JAKARTA
4,POL-0005,M-001,F,19810114,2015-08-28,JAKARTA


In [7]:
# Get min start date and max claim date to build a full timeline
min_date = polis_df['Tanggal Efektif Polis'].min()
claims_df['Tanggal Pasien Masuk RS'] = pd.to_datetime(claims_df['Tanggal Pasien Masuk RS'], errors='coerce')
max_date = claims_df['Tanggal Pasien Masuk RS'].max()

print(min_date)
print(max_date)

2011-12-05 00:00:00
2025-07-31 00:00:00


In [8]:
# Create a continuous monthly timeline and calculate cumulative active policies
all_months = pd.period_range(start=min_date, end=max_date, freq='M')
exposure_df = pd.DataFrame({'Period': all_months})
exposure_df.head()

,Period
0,2011-12
1,2012-01
2,2012-02
3,2012-03
4,2012-04


In [9]:
# Count how many new policies started each month
polis_df['Period'] = polis_df['Tanggal Efektif Polis'].dt.to_period('M')
monthly_sales = polis_df.groupby('Period').size().reset_index(name='New_Policies')
monthly_sales.head()

,Period,New_Policies
0,2011-12,42
1,2012-01,28
2,2012-02,46
3,2012-03,57
4,2012-04,41


In [10]:
# Merge timeline with sales and calculate cumulative exposure
exposure_df = pd.merge(exposure_df, monthly_sales, on='Period', how='left')
exposure_df['New_Policies'] = exposure_df['New_Policies'].fillna(0)
exposure_df['Exposure'] = exposure_df['New_Policies'].cumsum() # Total Active Policies
exposure_df.head()

,Period,New_Policies,Exposure
0,2011-12,42.0,42.0
1,2012-01,28.0,70.0
2,2012-02,46.0,116.0
3,2012-03,57.0,173.0
4,2012-04,41.0,214.0


In [11]:
# 3. PREPARE & SORT CLAIMS DATA
# Sort by Tanggal Pasien Masuk RS as requested
claims_df = claims_df.sort_values('Tanggal Pasien Masuk RS')
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.0,15025749.0,Indonesia
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.0,3400335.0,Indonesia
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.0,36417500.0,Indonesia
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.0,15667000.0,Indonesia
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.0,1870000.0,Indonesia


In [12]:
# Extract Month, Year, and Period for grouping
claims_df['Year'] = claims_df['Tanggal Pasien Masuk RS'].dt.year
claims_df['Month'] = claims_df['Tanggal Pasien Masuk RS'].dt.month
claims_df['Period'] = claims_df['Tanggal Pasien Masuk RS'].dt.to_period('M')
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS,Year,Month,Period
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.0,15025749.0,Indonesia,2024,1,2024-01
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.0,3400335.0,Indonesia,2024,1,2024-01
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.0,36417500.0,Indonesia,2024,1,2024-01
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.0,15667000.0,Indonesia,2024,1,2024-01
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.0,1870000.0,Indonesia,2024,1,2024-01


In [13]:
# Identify Inpatient (IP) vs Outpatient (OP, ODC, ODS)
claims_df['Is_IP'] = claims_df['Inpatient/Outpatient'].isin(['IP']).astype(int)
claims_df['Is_OP'] = claims_df['Inpatient/Outpatient'].isin(['OP', 'ODC', 'ODS']).astype(int)
claims_df.head()

,Claim ID,Nomor Polis,Reimburse/Cashless,Inpatient/Outpatient,ICD Diagnosis,ICD Description,Status Klaim,Tanggal Pembayaran Klaim,Tanggal Pasien Masuk RS,Tanggal Pasien Keluar RS,Nominal Klaim Yang Disetujui,Nominal Biaya RS Yang Terjadi,Lokasi RS,Year,Month,Period,Is_IP,Is_OP
2810,C-3627-M,POL-3872,C,IP,B34.2,"CORONAVIRUS INFECTION, UNSPECIFIED SITE",PAID,2024-04-25,2024-01-01,2024-01-07,14741310.0,15025749.0,Indonesia,2024,1,2024-01,1,0
3006,C-3836-M,POL-2078,C,OP,N18.5,"CHRONIC KIDNEY DISEASE, STAGE 5",PAID,2024-04-05,2024-01-02,2024-01-02,2769687.0,3400335.0,Indonesia,2024,1,2024-01,0,1
2816,C-3636-M,POL-3932,C,IP,H82,VERTIGINOUS SYNDROMES IN DISEASES CLASSIFIED E...,PAID,2024-02-06,2024-01-02,2024-01-07,36417500.0,36417500.0,Indonesia,2024,1,2024-01,1,0
1948,C-2519-M,POL-2436,R,IP,H35,OTHER RETINAL DISORDERS,PAID,2024-01-17,2024-01-02,2024-01-02,15667000.0,15667000.0,Indonesia,2024,1,2024-01,1,0
144,C-0245-M,POL-2200,C,OP,N23,UNSPECIFIED RENAL COLIC,PAID,2024-04-25,2024-01-02,2024-01-02,1839000.0,1870000.0,Indonesia,2024,1,2024-01,0,1


In [15]:
# 4. AGGREGATE PER MONTH
agg_df = claims_df.groupby(['Period', 'Year', 'Month']).agg(
    Jumlah_Inpatient=('Is_IP', 'sum'),
    Jumlah_Outpatient=('Is_OP', 'sum'),
    Jumlah_Klaim=('Claim ID', 'count'),
    Total_Nominal_Klaim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()
agg_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim
0,2024-01,2024,1,213,81,302,2.026098e+10
1,2024-02,2024,2,140,60,208,1.385965e+10
2,2024-03,2024,3,196,82,278,1.431126e+10
3,2024-04,2024,4,160,79,239,1.144106e+10
4,2024-05,2024,5,163,100,263,1.221146e+10


In [16]:
# 5. MERGE CLAIMS WITH EXPOSURE & CALCULATE METRICS
final_df = pd.merge(agg_df, exposure_df[['Period', 'Exposure']], on='Period', how='left')
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure
0,2024-01,2024,1,213,81,302,2.026098e+10,4096.0
1,2024-02,2024,2,140,60,208,1.385965e+10,4096.0
2,2024-03,2024,3,196,82,278,1.431126e+10,4096.0
3,2024-04,2024,4,160,79,239,1.144106e+10,4096.0
4,2024-05,2024,5,163,100,263,1.221146e+10,4096.0


In [17]:
# Metric 1: Ratio Inpatient / Outpatient (using np.where to avoid divide-by-zero errors)
final_df['Ratio Inpatient / Outpatient'] = np.where(
    final_df['Jumlah_Outpatient'] == 0, 
    np.nan, 
    final_df['Jumlah_Inpatient'] / final_df['Jumlah_Outpatient']
)
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient
0,2024-01,2024,1,213,81,302,2.026098e+10,4096.0,2.629630
1,2024-02,2024,2,140,60,208,1.385965e+10,4096.0,2.333333
2,2024-03,2024,3,196,82,278,1.431126e+10,4096.0,2.390244
3,2024-04,2024,4,160,79,239,1.144106e+10,4096.0,2.025316
4,2024-05,2024,5,163,100,263,1.221146e+10,4096.0,1.630000


In [18]:
# Metric 2: True Frequency Rate = (Jumlah Klaim / Active Policies)
final_df['Freq_Rate'] = final_df['Jumlah_Klaim'] / final_df['Exposure']
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient,Freq_Rate
0,2024-01,2024,1,213,81,302,2.026098e+10,4096.0,2.629630,0.073730
1,2024-02,2024,2,140,60,208,1.385965e+10,4096.0,2.333333,0.050781
2,2024-03,2024,3,196,82,278,1.431126e+10,4096.0,2.390244,0.067871
3,2024-04,2024,4,160,79,239,1.144106e+10,4096.0,2.025316,0.058350
4,2024-05,2024,5,163,100,263,1.221146e+10,4096.0,1.630000,0.064209


In [19]:
# Metric 3: Severity = (Total Nominal / Jumlah Klaim)
final_df['Severity'] = final_df['Total_Nominal_Klaim'] / final_df['Jumlah_Klaim']
final_df.head()

,Period,Year,Month,Jumlah_Inpatient,Jumlah_Outpatient,Jumlah_Klaim,Total_Nominal_Klaim,Exposure,Ratio Inpatient / Outpatient,Freq_Rate,Severity
0,2024-01,2024,1,213,81,302,2.026098e+10,4096.0,2.629630,0.073730,6.708934e+07
1,2024-02,2024,2,140,60,208,1.385965e+10,4096.0,2.333333,0.050781,6.663291e+07
2,2024-03,2024,3,196,82,278,1.431126e+10,4096.0,2.390244,0.067871,5.147935e+07
3,2024-04,2024,4,160,79,239,1.144106e+10,4096.0,2.025316,0.058350,4.787056e+07
4,2024-05,2024,5,163,100,263,1.221146e+10,4096.0,1.630000,0.064209,4.643141e+07


In [20]:
# 6. CLEAN UP AND RENAME COLUMNS
final_df = final_df[[
    'Period','Month', 'Year', 'Jumlah_Inpatient', 'Jumlah_Outpatient', 
    'Ratio Inpatient / Outpatient', 'Exposure', 'Jumlah_Klaim', 'Freq_Rate', 'Severity', 'Total_Nominal_Klaim'
]].copy()
final_df.head()

,Period,Month,Year,Jumlah_Inpatient,Jumlah_Outpatient,Ratio Inpatient / Outpatient,Exposure,Jumlah_Klaim,Freq_Rate,Severity,Total_Nominal_Klaim
0,2024-01,1,2024,213,81,2.629630,4096.0,302,0.073730,6.708934e+07,2.026098e+10
1,2024-02,2,2024,140,60,2.333333,4096.0,208,0.050781,6.663291e+07,1.385965e+10
2,2024-03,3,2024,196,82,2.390244,4096.0,278,0.067871,5.147935e+07,1.431126e+10
3,2024-04,4,2024,160,79,2.025316,4096.0,239,0.058350,4.787056e+07,1.144106e+10
4,2024-05,5,2024,163,100,1.630000,4096.0,263,0.064209,4.643141e+07,1.221146e+10


In [21]:
final_df = final_df.rename(columns={
    'Jumlah_Inpatient': 'Jumlah Inpatient',
    'Jumlah_Outpatient': 'Jumlah Outpatient',
    'Jumlah_Klaim': 'Frequency',
    'Total_Nominal_Klaim': 'Total Nominal Klaim'
})
final_df.head()

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim
0,2024-01,1,2024,213,81,2.629630,4096.0,302,0.073730,6.708934e+07,2.026098e+10
1,2024-02,2,2024,140,60,2.333333,4096.0,208,0.050781,6.663291e+07,1.385965e+10
2,2024-03,3,2024,196,82,2.390244,4096.0,278,0.067871,5.147935e+07,1.431126e+10
3,2024-04,4,2024,160,79,2.025316,4096.0,239,0.058350,4.787056e+07,1.144106e+10
4,2024-05,5,2024,163,100,1.630000,4096.0,263,0.064209,4.643141e+07,1.221146e+10


In [22]:

# Add Final "Total Klaim" column as requested
final_df['Total Klaim'] = final_df['Total Nominal Klaim']

# Remove any empty rows where date couldn't be parsed
final_df = final_df.dropna(subset=['Year', 'Month']).copy()
final_df['Year'] = final_df['Year'].astype(int)
final_df['Month'] = final_df['Month'].astype(int)
final_df.head()

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim,Total Klaim
0,2024-01,1,2024,213,81,2.629630,4096.0,302,0.073730,6.708934e+07,2.026098e+10,2.026098e+10
1,2024-02,2,2024,140,60,2.333333,4096.0,208,0.050781,6.663291e+07,1.385965e+10,1.385965e+10
2,2024-03,3,2024,196,82,2.390244,4096.0,278,0.067871,5.147935e+07,1.431126e+10,1.431126e+10
3,2024-04,4,2024,160,79,2.025316,4096.0,239,0.058350,4.787056e+07,1.144106e+10,1.144106e+10
4,2024-05,5,2024,163,100,1.630000,4096.0,263,0.064209,4.643141e+07,1.221146e+10,1.221146e+10


In [23]:
final_df

,Period,Month,Year,Jumlah Inpatient,Jumlah Outpatient,Ratio Inpatient / Outpatient,Exposure,Frequency,Freq_Rate,Severity,Total Nominal Klaim,Total Klaim
0,2024-01,1,2024,213,81,2.629630,4096.0,302,0.073730,6.708934e+07,2.026098e+10,2.026098e+10
1,2024-02,2,2024,140,60,2.333333,4096.0,208,0.050781,6.663291e+07,1.385965e+10,1.385965e+10
2,2024-03,3,2024,196,82,2.390244,4096.0,278,0.067871,5.147935e+07,1.431126e+10,1.431126e+10
3,2024-04,4,2024,160,79,2.025316,4096.0,239,0.058350,4.787056e+07,1.144106e+10,1.144106e+10
4,2024-05,5,2024,163,100,1.630000,4096.0,263,0.064209,4.643141e+07,1.221146e+10,1.221146e+10
5,2024-06,6,2024,106,119,0.890756,4096.0,225,0.054932,5.388963e+07,1.212517e+10,1.212517e+10
6,2024-07,7,2024,112,144,0.777778,4096.0,257,0.062744,5.825104e+07,1.497052e+10,1.497052e+10
7,2024-08,8,2024,95,130,0.730769,4096.0,228,0.055664,5.926726e+07,1.351294e+10,1.351294e+10
8,2024-09,9,2024,89,117,0.760684,4096.0,208,0.050781,5.896211e+07,1.226412e+10,1.226412e+10
9,2024-10,10,2024,107,164,0.652439,4096.0,274,0.066895,4.628163e+07,1.268117e+10,1.268117e+10


GBR

In [24]:
from sklearn.ensemble import GradientBoostingRegressor

In [25]:
def create_lag_features(data, target_col):
    df_feat = data.copy()
    # Lag Features (1, 2, 3 months back)
    for lag in [1, 2, 3]:
        df_feat[f'lag_{lag}'] = df_feat[target_col].shift(lag)
    # Rolling Feature (Mean of last 3 months)
    df_feat['rolling_mean_3'] = df_feat[target_col].shift(1).rolling(window=3).mean()
    return df_feat.dropna()

In [26]:
# 1. Train Model A - Frequency
train_freq = create_lag_features(final_df, 'Frequency')
X_freq = train_freq[['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']]
y_freq = train_freq['Frequency']

In [27]:
modelA_freq = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
modelA_freq.fit(X_freq, y_freq)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [28]:
# 2. Train Model A - Severity
train_sev = create_lag_features(final_df, 'Severity')
X_sev = train_sev[['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']]
y_sev = train_sev['Severity']

In [29]:
modelA_sev = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
modelA_sev.fit(X_sev, y_sev)

,loss,'squared_error'
,learning_rate,0.05
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [30]:
# --- STEP 3: RECURSIVE FORECASTING (Aug - Dec 2025) ---
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=final_df['Period'].max() + 1, end=target_end, freq='M')

In [31]:
# History DataFrame (starts with actual data, grows with predictions)
history_df = final_df.copy()
current_exposure = history_df['Exposure'].iloc[-1] # Assuming constant exposure for forecast

predictions = []

In [32]:
for month in forecast_range:
    # 1. Build Features from History (Last 3 rows)
    last_3 = history_df.tail(3)
    
    # Feature Vector for Frequency
    freq_feats = np.array([[
        last_3['Frequency'].iloc[-1], # lag_1
        last_3['Frequency'].iloc[-2], # lag_2
        last_3['Frequency'].iloc[-3], # lag_3
        last_3['Frequency'].mean()    # rolling_mean_3
    ]])
    
    # Feature Vector for Severity
    sev_feats = np.array([[
        last_3['Severity'].iloc[-1],
        last_3['Severity'].iloc[-2],
        last_3['Severity'].iloc[-3],
        last_3['Severity'].mean()
    ]])
    
    # 2. Predict
    pred_frequency = modelA_freq.predict(freq_feats)[0]
    pred_sev = modelA_sev.predict(sev_feats)[0]
    
    # Safety: Ensure no negative predictions
    pred_frequency = max(0, pred_frequency)
    pred_sev = max(0, pred_sev)
    
    # 3. Calculate Derived Metrics
    pred_claim_count = pred_frequency
    pred_total_claim = pred_claim_count * pred_sev
    
    # 4. Append to History
    new_row = pd.DataFrame([{
        'Month_Period': month,
        'Claim_Count': pred_claim_count,
        'Total_Nominal': pred_total_claim,
        'Exposure': current_exposure,
        'Frequency': pred_frequency,
        'Severity': pred_sev
    }])
    history_df = pd.concat([history_df, new_row], ignore_index=True)
    
    # 5. Store if in Target Range (Aug - Dec)
    if month >= target_start:
        predictions.append({
            'Month': str(month),
            'Jumlah Klaim': int(round(pred_claim_count)),
            'Total Klaim': pred_total_claim,
            'Frequency': pred_frequency,
            'Severity': pred_sev
        })

c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\va

In [33]:
# --- STEP 4: FINAL RESULT ---
modelA_result_df = pd.DataFrame(predictions)

# Formatting for Display
pd.options.display.float_format = '{:f}'.format
print("Model A Prediction Results (Aug - Dec 2025):")
print(modelA_result_df)

Model A Prediction Results (Aug - Dec 2025):
     Month  Jumlah Klaim        Total Klaim  Frequency        Severity
0  2025-08           224 13006524431.265368 224.166173 58021798.163976
1  2025-09           252 12724009375.474773 252.393986 50413282.761677
2  2025-10           228 12129944638.245440 228.127103 53171869.905693
3  2025-11           209 11091678351.623657 208.801263 53120743.572480
4  2025-12           272 15872474142.575233 272.252172 58300633.602397


In [34]:
# Create a list to store the formatted rows
formatted_data = []

for index, row in modelA_result_df.iterrows():
    # Format the month from '2025-08' to '2025_08'
    month_str = str(row['Month']).replace('-', '_')
    
    # Append Frequency
    formatted_data.append({
        'id': f"{month_str}_Claim_Frequency",
        'value': row['Frequency']
    })
    
    # Append Severity
    formatted_data.append({
        'id': f"{month_str}_Claim_Severity",
        'value': row['Severity']
    })
    
    # Append Total Claim
    formatted_data.append({
        'id': f"{month_str}_Total_Claim",
        'value': row['Total Klaim']
    })

# Create the final dataframe
submission_df = pd.DataFrame(formatted_data)

# Display
print(submission_df)

# Export to CSV without the index number
submission_df.to_csv('submission_gb.csv', index=False)

print("File 'submission_gb.csv' saved successfully.")

                         id              value
0   2025_08_Claim_Frequency         224.166173
1    2025_08_Claim_Severity    58021798.163976
2       2025_08_Total_Claim 13006524431.265368
3   2025_09_Claim_Frequency         252.393986
4    2025_09_Claim_Severity    50413282.761677
5       2025_09_Total_Claim 12724009375.474773
6   2025_10_Claim_Frequency         228.127103
7    2025_10_Claim_Severity    53171869.905693
8       2025_10_Total_Claim 12129944638.245440
9   2025_11_Claim_Frequency         208.801263
10   2025_11_Claim_Severity    53120743.572480
11      2025_11_Total_Claim 11091678351.623657
12  2025_12_Claim_Frequency         272.252172
13   2025_12_Claim_Severity    58300633.602397
14      2025_12_Total_Claim 15872474142.575233
File 'submission_gb.csv' saved successfully.


RandomForest

In [35]:
from sklearn.ensemble import RandomForestRegressor

In [36]:
# --- 1. PREPARE DATA ---
# Ensure strict ordering
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

In [37]:
# Create Lag Features (Shift the data to use past values to predict future)
for col in ['Freq_Rate', 'Severity']:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    df[f'{col}_Mean3'] = df[col].rolling(window=3).mean()

In [38]:
# Drop NaN values created by shifting (first 3 rows)
train_df = df.dropna().copy()

# Define Features and Targets
features = ['Month', 'Freq_Rate_Lag1', 'Freq_Rate_Lag2', 'Freq_Rate_Lag3', 'Freq_Rate_Mean3', 
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_Mean3']

In [39]:
# Model 1: Frequency (Count)
X = train_df[features]
y_freq = train_df['Frequency'] # Target is now the Count
model_freq = RandomForestRegressor(n_estimators=100, random_state=42)
model_freq.fit(X, y_freq)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [40]:
# Model 2: Severity
y_sev = train_df['Severity']
model_sev = RandomForestRegressor(n_estimators=100, random_state=42)
model_sev.fit(X, y_sev)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [41]:
# --- 2. FORECASTING LOOP ---

# Setup Forecast Dates
last_date_in_data = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')

# Generate range from next month after data ends -> until Dec 2025
forecast_range = pd.period_range(start=last_date_in_data + 1, end=target_end, freq='M')

# Initialize current_history with original data
current_history = df.copy()
predictions = []

print(f"Starting forecast from {forecast_range[0]} to {forecast_range[-1]}...")

for period in forecast_range:
    # 1. Extract Features from the most recent 'current_history'
    last_3 = current_history.tail(3)
    
    # Build feature vector matching the training features
    input_features = np.array([[
        period.month,                            # Month (Seasonality)
        last_3['Frequency'].iloc[-1],            # Lag 1 (Count)
        last_3['Frequency'].iloc[-2],            # Lag 2 (Count)
        last_3['Frequency'].iloc[-3],            # Lag 3 (Count)
        last_3['Frequency'].mean(),              # Rolling Mean (Count)
        last_3['Severity'].iloc[-1],             # Lag 1 Sev
        last_3['Severity'].iloc[-2],             # Lag 2 Sev
        last_3['Severity'].iloc[-3],             # Lag 3 Sev
        last_3['Severity'].mean()                # Rolling Mean Sev
    ]])
    
    # 2. Predict
    pred_frequency_count = model_freq.predict(input_features)[0] # Output is Count
    pred_severity = model_sev.predict(input_features)[0]
    
    # Safety: Ensure no negative predictions
    pred_frequency_count = max(0, pred_frequency_count)
    pred_severity = max(0, pred_severity)

    # 3. Calculate Total Claim
    # Total Claim = Count * Average Cost per Claim
    pred_total_claim = pred_frequency_count * pred_severity
    
    # 4. Save Prediction if inside our target window (Aug-Dec)
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_frequency_count,
            'Severity': pred_severity,
            'Total_Claim': pred_total_claim
        })
    
    # 5. Append prediction to history so next month uses it as "past data"
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_frequency_count, # Store predicted count
        'Severity': pred_severity,
        # Fill other cols like Exposure with 0 or NaN as they aren't used in features anymore
    }])
    
    current_history = pd.concat([current_history, new_row], ignore_index=True)

# Convert results to DataFrame
results_df = pd.DataFrame(predictions)
print("\nForecast Results (Aug - Dec 2025):")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])

Starting forecast from 2025-08 to 2025-12...

Forecast Results (Aug - Dec 2025):
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 243.170000 52361274.850084 12732691205.294868
1      2025-09 246.990000 51332984.216623 12678733771.663691
2      2025-10 245.720000 52089151.725578 12799346362.009136
3      2025-11 245.640000 52191767.647221 12820385804.863464
4      2025-12 243.930000 52127275.780670 12715406381.178873


c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749

In [42]:
# --- 3. FORMAT FOR SUBMISSION ---
formatted_data = []

for index, row in results_df.iterrows():
    month_id = row['id_month']
    
    # 1. Frequency (Count)
    formatted_data.append({
        'id': f"{month_id}_Claim_Frequency",
        'value': row['Frequency']
    })
    
    # 2. Severity
    formatted_data.append({
        'id': f"{month_id}_Claim_Severity",
        'value': row['Severity']
    })
    
    # 3. Total Claim
    formatted_data.append({
        'id': f"{month_id}_Total_Claim",
        'value': row['Total_Claim']
    })

submission_df = pd.DataFrame(formatted_data)

# Export
submission_df.to_csv('submission_rf.csv', index=False)
print("\nSaved to 'submission_rf.csv'")


Saved to 'submission_rf.csv'


XGBoost

In [43]:
from xgboost import XGBRegressor

In [44]:
# --- 1. SETUP & PREPROCESSING ---
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# A. Create Cyclical Month Features (Crucial for Seasonality)
# This helps the model understand that Month 12 is close to Month 1
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

In [45]:
# B. Create Lag & Rolling Features
# We do this for both targets: Frequency (Count) and Severity
targets = ['Frequency', 'Severity']

for col in targets:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()

In [46]:
# Drop the first 3 rows which now have NaNs
train_df = df.dropna().copy()

# --- 2. DEFINE FEATURES & LOG-TRANSFORM TARGETS ---
features = ['month_sin', 'month_cos', 
            'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_RollMean3',
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_RollMean3']

# Apply Log-Transformation to targets to handle skewness and prevent negative predictions
# log1p = log(x + 1) to handle zeros safely
y_freq_log = np.log1p(train_df['Frequency'])
y_sev_log = np.log1p(train_df['Severity'])

X = train_df[features]

In [47]:
# --- 3. TRAIN XGBOOST MODELS ---
# Using robust parameters to prevent overfitting
params = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'n_jobs': -1,
    'random_state': 42
}

print("Training Frequency Model...")
model_freq = XGBRegressor(**params)
model_freq.fit(X, y_freq_log)

print("Training Severity Model...")
model_sev = XGBRegressor(**params)
model_sev.fit(X, y_sev_log)

print("Training Complete.")

Training Frequency Model...
Training Severity Model...
Training Complete.


In [48]:
# --- 4. RECURSIVE FORECASTING (Aug - Dec 2025) ---

# Setup Dates
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

# Initialize History
current_history = df.copy()
predictions = []

print(f"\nForecasting from {forecast_range[0]} to {forecast_range[-1]}...")

for period in forecast_range:
    # A. Prepare Input Features from History
    last_3 = current_history.tail(3)
    
    # Calculate Cyclic Month for the forecast period
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    input_row = np.array([[
        feat_sin,
        feat_cos,
        last_3['Frequency'].iloc[-1],      # Lag1
        last_3['Frequency'].iloc[-2],      # Lag2
        last_3['Frequency'].iloc[-3],      # Lag3
        last_3['Frequency'].mean(),        # Mean3
        last_3['Severity'].iloc[-1],       # Lag1 Sev
        last_3['Severity'].iloc[-2],       # Lag2 Sev
        last_3['Severity'].iloc[-3],       # Lag3 Sev
        last_3['Severity'].mean()          # Mean3 Sev
    ]])
    
    # B. Predict (Result is in Log Scale)
    pred_freq_log = model_freq.predict(input_row)[0]
    pred_sev_log = model_sev.predict(input_row)[0]
    
    # C. Inverse Transform (Convert back from Log to Real numbers)
    pred_freq = np.expm1(pred_freq_log)
    pred_sev = np.expm1(pred_sev_log)
    
    # D. Calculate Total Claim
    pred_total_claim = pred_freq * pred_sev
    
    # E. Store & Update History
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    # Add new row to history for next iteration
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_freq,
        'Severity': pred_sev,
        # Recalculate cyclic features for the history df if needed, though only raw cols are used for lags
        'month_sin': feat_sin,
        'month_cos': feat_cos
    }])
    
    current_history = pd.concat([current_history, new_row], ignore_index=True)

results_df = pd.DataFrame(predictions)

# Show Quick Preview
print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])


Forecasting from 2025-08 to 2025-12...

Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 260.144257 53604112.000000 13944802304.000000
1      2025-09 250.288895 52227120.000000 13071867904.000000
2      2025-10 256.413696 50671360.000000 12992830464.000000
3      2025-11 255.200531 50869876.000000 12982019072.000000
4      2025-12 234.500229 53290840.000000 12496713728.000000


In [49]:
# --- 5. FORMAT AND EXPORT ---
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    
    # Frequency
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    # Severity
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    # Total Claim
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_xgboost.csv', index=False)

print("\nSuccessfully saved to 'submission_xgboost.csv'")


Successfully saved to 'submission_xgboost.csv'


LightGBM

In [50]:
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_percentage_error

c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
# --- 1. SETUP & PREPROCESSING ---
# Ensure strict ordering
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# A. Create Cyclical Month Features (Seasonality)
# Transforms month 1-12 into coordinates on a circle so Dec (12) is close to Jan (1)
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

In [52]:
# B. Create Lag & Rolling Features
targets = ['Frequency', 'Severity']

for col in targets:
    # Immediate past values
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    # Trend over the last window
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()

# Drop rows with NaN (created by lags) for training
train_df = df.dropna().copy()

In [53]:
# --- 2. DEFINE FEATURES & LOG-TRANSFORM TARGETS ---
features = ['month_sin', 'month_cos', 
            'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_RollMean3',
            'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_RollMean3']

# Log-Transform Targets: Handles skewness & prevents negative predictions
y_freq_log = np.log1p(train_df['Frequency'])
y_sev_log = np.log1p(train_df['Severity'])

X = train_df[features]

In [54]:
# --- 3. TRAIN LIGHTGBM MODELS ---
# Parameters optimized for small tabular data to prevent overfitting
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 500,       # Number of trees
    'learning_rate': 0.05,     # Step size (smaller is safer)
    'num_leaves': 31,          # Max leaves per tree
    'max_depth': -1,           # No limit (let num_leaves control complexity)
    'min_child_samples': 5,    # Important for small data: min data in a leaf
    'subsample': 0.8,          # Use 80% of rows per iteration
    'colsample_bytree': 0.8,   # Use 80% of features per iteration
    'random_state': 42,
    'verbose': -1              # Silence warnings
}

In [55]:
print("Training Frequency (Count) Model...")
model_freq = lgb.LGBMRegressor(**params)
model_freq.fit(X, y_freq_log)

Training Frequency (Count) Model...


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [56]:
print("Training Severity Model...")
model_sev = lgb.LGBMRegressor(**params)
model_sev.fit(X, y_sev_log)

Training Severity Model...


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


In [57]:
# --- 4. RECURSIVE FORECASTING (Aug - Dec 2025) ---

# Define Time Range
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

# Initialize History with original data
current_history = df.copy()
predictions = []

print(f"\nForecasting from {forecast_range[0]} to {forecast_range[-1]}...")

for period in forecast_range:
    # A. Build Input Row from History (Last 3 months)
    last_3 = current_history.tail(3)
    
    # Calculate Cyclic Month for the FORECAST month
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    # Construct Feature Array (must match training order)
    input_row = np.array([[
        feat_sin,
        feat_cos,
        last_3['Frequency'].iloc[-1],      # Lag1
        last_3['Frequency'].iloc[-2],      # Lag2
        last_3['Frequency'].iloc[-3],      # Lag3
        last_3['Frequency'].mean(),        # RollMean3
        last_3['Severity'].iloc[-1],       # Lag1 Sev
        last_3['Severity'].iloc[-2],       # Lag2 Sev
        last_3['Severity'].iloc[-3],       # Lag3 Sev
        last_3['Severity'].mean()          # RollMean3 Sev
    ]])
    
    # B. Predict (Log Scale)
    pred_freq_log = model_freq.predict(input_row)[0]
    pred_sev_log = model_sev.predict(input_row)[0]
    
    # C. Inverse Transform (Back to Real Scale)
    pred_freq = np.expm1(pred_freq_log)
    pred_sev = np.expm1(pred_sev_log)
    
    # D. Calculate Total Claim (Count * Severity)
    pred_total_claim = pred_freq * pred_sev
    
    # E. Store Prediction (Only if in Aug-Dec window)
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    # F. Update History for Next Iteration
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_freq,
        'Severity': pred_sev,
        # Helper cols (not strictly needed as we rebuild them, but good for consistency)
        'month_sin': feat_sin,
        'month_cos': feat_cos
    }])
    
    current_history = pd.concat([current_history, new_row], ignore_index=True)

# Create Results DataFrame
results_df = pd.DataFrame(predictions)

# Show Preview
print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])


Forecasting from 2025-08 to 2025-12...

Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 233.816257 52797448.293320 12344901728.604944
1      2025-09 247.148268 48833481.821214 12069110457.186052
2      2025-10 232.643995 54516762.430975 12682997398.343206
3      2025-11 220.987987 48841204.159681 10793319398.009361
4      2025-12 237.003254 60426164.856133 14321197670.333136


c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have v

In [58]:
# --- 5. FORMAT AND EXPORT ---
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    
    # Frequency
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    # Severity
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    # Total Claim
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_lightgbm.csv', index=False)

print("\nSuccessfully saved to 'submission_lightgbm.csv'")


Successfully saved to 'submission_lightgbm.csv'


Improved Light GBM

In [59]:
# --- 1. ENHANCED FEATURE ENGINEERING ---
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# A. Cyclical Month
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

# B. Expanded Lags & Rolling Windows
targets = ['Frequency', 'Severity']

for col in targets:
    # Lags
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    
    # Rolling Means (Short vs Long term trend)
    df[f'{col}_Mean3'] = df[col].rolling(window=3).mean()
    df[f'{col}_Mean6'] = df[col].rolling(window=6).mean() # New: Half-year trend
    
    # Rolling Volatility (Standard Deviation)
    df[f'{col}_Std3'] = df[col].rolling(window=3).std()   # New: Is data becoming unstable?

# Drop NaNs
train_df = df.dropna().copy()

features = [c for c in train_df.columns if 'Lag' in c or 'Mean' in c or 'Std' in c or 'month_' in c]
X = train_df[features]

# We train on Log Scale to handle skew, but optimize for MAPE
y_freq = train_df['Frequency']
y_sev = train_df['Severity']

# --- 2. OPTUNA OPTIMIZATION FUNCTION ---

def optimize_lightgbm(target_col, n_trials=50):
    print(f"\n--- Optimizing {target_col} ---")
    y = train_df[target_col]
    
    # Time-Series Split for Validation (Train on first 80%, Validate on last 20%)
    split_idx = int(len(X) * 0.8)
    X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]
    
    def objective(trial):
        params = {
            'objective': 'mape', # Directly optimize MAPE!
            'metric': 'mape',
            'verbosity': -1,
            'boosting_type': 'gbdt',
            'random_state': 42,
            'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 2, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        
        # Calculate MAPE
        mape = mean_absolute_percentage_error(y_val, preds)
        return mape

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    
    print(f"Best MAPE for {target_col}: {study.best_value:.4f}")
    return study.best_params

# --- 3. RUN OPTIMIZATION ---
# Run 50 trials for each model to find the perfect parameters
best_params_freq = optimize_lightgbm('Frequency', n_trials=50)
best_params_sev = optimize_lightgbm('Severity', n_trials=50)

# --- 4. TRAIN FINAL MODELS WITH BEST PARAMS ---
print("\nTraining Final Models...")
final_model_freq = lgb.LGBMRegressor(**best_params_freq)
final_model_freq.fit(X, y_freq)

final_model_sev = lgb.LGBMRegressor(**best_params_sev)
final_model_sev.fit(X, y_sev)

[I 2026-02-22 20:15:01,038] A new study created in memory with name: no-name-5034c6e1-cd6b-41ab-b7ba-ccd383ee35b7
[I 2026-02-22 20:15:01,054] Trial 0 finished with value: 0.06117959988252875 and parameters: {'n_estimators': 709, 'learning_rate': 0.01908107988602684, 'num_leaves': 84, 'max_depth': 10, 'min_child_samples': 16, 'subsample': 0.8515922968667107, 'colsample_bytree': 0.9566001592947428, 'reg_alpha': 0.10667583639736908, 'reg_lambda': 6.1424303167034584e-06}. Best is trial 0 with value: 0.06117959988252875.
[I 2026-02-22 20:15:01,070] Trial 1 finished with value: 0.06117959988252875 and parameters: {'n_estimators': 482, 'learning_rate': 0.016560148313955775, 'num_leaves': 39, 'max_depth': 7, 'min_child_samples': 8, 'subsample': 0.7409583120949748, 'colsample_bytree': 0.6550163210489961, 'reg_alpha': 0.5408029503050257, 'reg_lambda': 0.0010427935054212291}. Best is trial 0 with value: 0.06117959988252875.
[I 2026-02-22 20:15:01,079] Trial 2 finished with value: 0.06117959988252


--- Optimizing Frequency ---


[I 2026-02-22 20:15:01,107] Trial 5 finished with value: 0.06117959988252875 and parameters: {'n_estimators': 278, 'learning_rate': 0.020139711961973675, 'num_leaves': 33, 'max_depth': 9, 'min_child_samples': 15, 'subsample': 0.7783714455106039, 'colsample_bytree': 0.5138858458859059, 'reg_alpha': 2.1842662470882914e-06, 'reg_lambda': 1.4473998430297468}. Best is trial 0 with value: 0.06117959988252875.
[I 2026-02-22 20:15:01,116] Trial 6 finished with value: 0.06117959988252875 and parameters: {'n_estimators': 626, 'learning_rate': 0.020267228323628577, 'num_leaves': 78, 'max_depth': 12, 'min_child_samples': 13, 'subsample': 0.5857809923973682, 'colsample_bytree': 0.6907954424916825, 'reg_alpha': 0.0008555045140102968, 'reg_lambda': 2.2231834510581387e-07}. Best is trial 0 with value: 0.06117959988252875.
[I 2026-02-22 20:15:01,126] Trial 7 finished with value: 0.06117959988252875 and parameters: {'n_estimators': 327, 'learning_rate': 0.0575359840929628, 'num_leaves': 62, 'max_depth':

Best MAPE for Frequency: 0.0612

--- Optimizing Severity ---


[I 2026-02-22 20:15:02,569] Trial 14 finished with value: 0.04812267077268912 and parameters: {'n_estimators': 271, 'learning_rate': 0.06221614950055737, 'num_leaves': 77, 'max_depth': 6, 'min_child_samples': 3, 'subsample': 0.7175696216786838, 'colsample_bytree': 0.7531473740734466, 'reg_alpha': 0.03973702283496695, 'reg_lambda': 0.003439289703536379}. Best is trial 0 with value: 0.04812267077268912.
[I 2026-02-22 20:15:02,589] Trial 15 finished with value: 0.04812267077268912 and parameters: {'n_estimators': 460, 'learning_rate': 0.03464720884586533, 'num_leaves': 56, 'max_depth': 9, 'min_child_samples': 16, 'subsample': 0.5820235579648914, 'colsample_bytree': 0.6911308160300118, 'reg_alpha': 6.144183987768658e-07, 'reg_lambda': 3.6068678286209763e-06}. Best is trial 0 with value: 0.04812267077268912.
[I 2026-02-22 20:15:02,614] Trial 16 finished with value: 0.04812267077268912 and parameters: {'n_estimators': 752, 'learning_rate': 0.02132683008758179, 'num_leaves': 37, 'max_depth': 

Best MAPE for Severity: 0.0481

Training Final Models...


,boosting_type,'gbdt'
,num_leaves,49
,max_depth,7
,learning_rate,0.0842085563039455
,n_estimators,130
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,17


In [60]:
# --- 5. RECURSIVE FORECASTING ---
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

current_history = df.copy()
predictions = []

for period in forecast_range:
    # A. Build Features
    last_3 = current_history.tail(3)
    last_6 = current_history.tail(6) # Need 6 rows for Mean6
    
    # Cyclic Month
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    input_row = pd.DataFrame([{
        'month_sin': feat_sin,
        'month_cos': feat_cos,
        'Frequency_Lag1': last_3['Frequency'].iloc[-1],
        'Frequency_Lag2': last_3['Frequency'].iloc[-2],
        'Frequency_Lag3': last_3['Frequency'].iloc[-3],
        'Frequency_Mean3': last_3['Frequency'].mean(),
        'Frequency_Mean6': last_6['Frequency'].mean() if len(last_6)>=6 else last_3['Frequency'].mean(),
        'Frequency_Std3': last_3['Frequency'].std(),
        
        'Severity_Lag1': last_3['Severity'].iloc[-1],
        'Severity_Lag2': last_3['Severity'].iloc[-2],
        'Severity_Lag3': last_3['Severity'].iloc[-3],
        'Severity_Mean3': last_3['Severity'].mean(),
        'Severity_Mean6': last_6['Severity'].mean() if len(last_6)>=6 else last_3['Severity'].mean(),
        'Severity_Std3': last_3['Severity'].std(),
    }])
    
    # Ensure column order matches training
    input_row = input_row[features]
    
    # B. Predict
    # Note: We optimized on raw data (not log), so we predict raw directly
    pred_freq = final_model_freq.predict(input_row)[0]
    pred_sev = final_model_sev.predict(input_row)[0]
    
    # Safety Capping (Frequency shouldn't be negative)
    pred_freq = max(0, pred_freq)
    pred_sev = max(0, pred_sev)
    
    pred_total_claim = pred_freq * pred_sev
    
    # C. Store & Update
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_freq,
        'Severity': pred_sev
        # Features will be recalculated in next loop
    }])
    
    current_history = pd.concat([current_history, new_row], ignore_index=True)

# Export
results_df = pd.DataFrame(predictions)
formatted_data = []
for index, row in results_df.iterrows():
    m_id = row['id_month']
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

pd.DataFrame(formatted_data).to_csv('submission_optuna_lgbm.csv', index=False)
print("Optimized submission saved!")

Optimized submission saved!


LightGBM Using Lag 6 and Lag 12

In [61]:
# --- 1. SETUP & FEATURE ENGINEERING ---
# Ensure strict ordering
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# Create Cyclical Month Features
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

targets = ['Frequency', 'Severity']

for col in targets:
    # Short-term lags
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    
    # Long-term / Seasonal lags (Your suggestion)
    df[f'{col}_Lag6'] = df[col].shift(6)
    df[f'{col}_Lag12'] = df[col].shift(12)
    
    # Rolling trends
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()
    df[f'{col}_RollMean6'] = df[col].rolling(window=6).mean()

# IMPORTANT: We do NOT use dropna() here. 
# LightGBM can handle NaNs in the early rows natively.
train_df = df.copy()

# Define Features and Log-Transform Targets
features = [c for c in train_df.columns if 'Lag' in c or 'Roll' in c or 'month_' in c]

y_freq_log = np.log1p(train_df['Frequency'])
y_sev_log = np.log1p(train_df['Severity'])

X = train_df[features]

# --- 2. TRAIN ROBUST LIGHTGBM ---
# Reverting to the simpler parameters that got you 9.48%, avoiding Optuna overfitting
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 300,       # Kept reasonable to prevent overfitting
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 3,    # Lowered slightly to allow learning from smaller data
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

print("Training Frequency Model with Lag 6 & 12...")
model_freq = lgb.LGBMRegressor(**params)
model_freq.fit(X, y_freq_log)

print("Training Severity Model with Lag 6 & 12...")
model_sev = lgb.LGBMRegressor(**params)
model_sev.fit(X, y_sev_log)

print("Training Complete.")

Training Frequency Model with Lag 6 & 12...
Training Severity Model with Lag 6 & 12...
Training Complete.


In [62]:
# --- 3. RECURSIVE FORECASTING (Aug - Dec 2025) ---
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

current_history = df.copy()
predictions = []

print(f"\nForecasting from {forecast_range[0]} to {forecast_range[-1]}...")

for period in forecast_range:
    # Cyclic Month
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    # Safely extract lags (if history is too short for some reason, use NaN)
    def get_lag(col, n):
        if len(current_history) >= n:
            return current_history[col].iloc[-n]
        return np.nan

    def get_roll_mean(col, n):
        if len(current_history) >= n:
            return current_history[col].tail(n).mean()
        return current_history[col].mean()

    # Construct Feature Dictionary matching training features
    input_dict = {
        'month_sin': feat_sin,
        'month_cos': feat_cos,
        
        'Frequency_Lag1': get_lag('Frequency', 1),
        'Frequency_Lag2': get_lag('Frequency', 2),
        'Frequency_Lag3': get_lag('Frequency', 3),
        'Frequency_Lag6': get_lag('Frequency', 6),
        'Frequency_Lag12': get_lag('Frequency', 12),
        'Frequency_RollMean3': get_roll_mean('Frequency', 3),
        'Frequency_RollMean6': get_roll_mean('Frequency', 6),
        
        'Severity_Lag1': get_lag('Severity', 1),
        'Severity_Lag2': get_lag('Severity', 2),
        'Severity_Lag3': get_lag('Severity', 3),
        'Severity_Lag6': get_lag('Severity', 6),
        'Severity_Lag12': get_lag('Severity', 12),
        'Severity_RollMean3': get_roll_mean('Severity', 3),
        'Severity_RollMean6': get_roll_mean('Severity', 6),
    }
    
    # Convert to DataFrame to ensure correct column order
    input_row = pd.DataFrame([input_dict])[features]
    
    # Predict (Inverse Log Transform)
    pred_freq = np.expm1(model_freq.predict(input_row)[0])
    pred_sev = np.expm1(model_sev.predict(input_row)[0])
    
    # Calculate Total Claim
    pred_total_claim = pred_freq * pred_sev
    
    # Store Prediction
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    # Update History for next step
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_freq,
        'Severity': pred_sev
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)

results_df = pd.DataFrame(predictions)
print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])


Forecasting from 2025-08 to 2025-12...

Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 249.203942 55758378.477251 13895207709.053745
1      2025-09 248.645809 55589285.637824 13822042887.137014
2      2025-10 242.784833 49440718.891631 12003456664.984371
3      2025-11 245.368014 51542662.671333 12646920777.091873
4      2025-12 232.962684 54713340.244874 12746166617.062954


In [63]:
# --- 4. FORMAT FOR SUBMISSION ---
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_lgbm_lag12.csv', index=False)

print("\nSuccessfully saved to 'submission_lgbm_lag12.csv'")


Successfully saved to 'submission_lgbm_lag12.csv'


GLM Training

In [64]:
import pandas as pd
import numpy as np
from sklearn.linear_model import PoissonRegressor, GammaRegressor

# --- 1. SETUP & FEATURE ENGINEERING ---
# Ensure strict ordering
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# Create a Time Index for the overall upward/downward trend (0, 1, 2, 3...)
df['Time_Index'] = np.arange(len(df))

# Create Cyclical Month Features for the recurring wave (Seasonality)
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

# Features: We ONLY use Time and Seasonality. No lags! No lost data!
features = ['Time_Index', 'month_sin', 'month_cos']
X = df[features]

# Targets
y_freq = df['Frequency']

# Gamma requires strictly positive values (>0). 
# If there are months with 0 severity, we add a tiny fraction to prevent errors.
y_sev = df['Severity'].apply(lambda x: x if x > 0 else 1e-6)

# --- 2. TRAIN GLMs ---
# Poisson for Frequency (Counts)
print("Training Poisson GLM for Frequency...")
model_freq = PoissonRegressor(alpha=1e-4, max_iter=1000)
model_freq.fit(X, y_freq)

# Gamma for Severity (Costs)
print("Training Gamma GLM for Severity...")
model_sev = GammaRegressor(alpha=1e-4, max_iter=1000)
model_sev.fit(X, y_sev)

print("Training Complete.")

Training Poisson GLM for Frequency...
Training Gamma GLM for Severity...
Training Complete.


In [65]:
# --- 3. FORECASTING (Aug - Dec 2025) ---
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

# The next Time_Index continues from where the training data ended
next_time_index = df['Time_Index'].max() + 1
predictions = []

print(f"\nForecasting from {forecast_range[0]} to {forecast_range[-1]}...")

for period in forecast_range:
    p_month = period.month
    
    # Build the future feature row
    input_row = pd.DataFrame([{
        'Time_Index': next_time_index,
        'month_sin': np.sin(2 * np.pi * p_month/12),
        'month_cos': np.cos(2 * np.pi * p_month/12)
    }])
    
    # Predict directly
    pred_freq = model_freq.predict(input_row)[0]
    pred_sev = model_sev.predict(input_row)[0]
    
    # Calculate Total Claim
    pred_total_claim = pred_freq * pred_sev
    
    # Store Prediction
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    next_time_index += 1 # Advance the clock for the next month

results_df = pd.DataFrame(predictions)
print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])


Forecasting from 2025-08 to 2025-12...

Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 234.417536 51915705.703318 12169951792.539413
1      2025-09 236.005133 52136989.569535 12304597154.549131
2      2025-10 236.384220 52690071.936137 12455101531.121786
3      2025-11 235.131964 53396696.292686 12555270079.109734
4      2025-12 232.293137 54031567.432815 12551162278.798311


In [66]:
# --- 4. FORMAT FOR SUBMISSION ---
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_glm.csv', index=False)

print("\nSuccessfully saved to 'submission_glm.csv'")


Successfully saved to 'submission_glm.csv'


Light GBM + GLM Training

In [67]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import PoissonRegressor, GammaRegressor

# --- 1. SETUP & FEATURE ENGINEERING ---
# Ensure strict ordering
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# A. Features for GLM (Trend & Seasonality)
df['Time_Index'] = np.arange(len(df))
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

# B. Features for LightGBM (Lags & Rolling Means)
targets = ['Frequency', 'Severity']
for col in targets:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()

# Separate training sets
# LightGBM needs rows with lags (drops first 3)
train_lgbm = df.dropna().copy()
features_lgbm = [c for c in train_lgbm.columns if 'Lag' in c or 'Roll' in c or 'month_' in c]

# GLM uses all rows (no lags needed)
features_glm = ['Time_Index', 'month_sin', 'month_cos']

# --- 2. TRAIN THE MODELS ---

# A. Train LightGBM (Using the settings from your 9.48% run)
y_freq_log = np.log1p(train_lgbm['Frequency'])
y_sev_log = np.log1p(train_lgbm['Severity'])

lgbm_params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 31,
    'min_child_samples': 5, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'random_state': 42, 'verbose': -1
}

print("Training LightGBM Models...")
model_freq_lgbm = lgb.LGBMRegressor(**lgbm_params).fit(train_lgbm[features_lgbm], y_freq_log)
model_sev_lgbm = lgb.LGBMRegressor(**lgbm_params).fit(train_lgbm[features_lgbm], y_sev_log)

# B. Train GLM
print("Training GLM Models...")
y_freq_glm = df['Frequency']
y_sev_glm = df['Severity'].apply(lambda x: x if x > 0 else 1e-6)

model_freq_glm = PoissonRegressor(alpha=1e-4, max_iter=1000).fit(df[features_glm], y_freq_glm)
model_sev_glm = GammaRegressor(alpha=1e-4, max_iter=1000).fit(df[features_glm], y_sev_glm)

print("Training Complete. Proceeding to Hybrid Forecast.")

# --- 3. RECURSIVE HYBRID FORECASTING (Aug - Dec 2025) ---
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

current_history = df.copy()
next_time_index = df['Time_Index'].max() + 1
predictions = []

for period in forecast_range:
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    
    # 1. Prepare LightGBM Inputs (from history)
    last_3 = current_history.tail(3)
    input_lgbm = pd.DataFrame([{
        'month_sin': feat_sin, 'month_cos': feat_cos,
        'Frequency_Lag1': last_3['Frequency'].iloc[-1],
        'Frequency_Lag2': last_3['Frequency'].iloc[-2],
        'Frequency_Lag3': last_3['Frequency'].iloc[-3],
        'Frequency_RollMean3': last_3['Frequency'].mean(),
        'Severity_Lag1': last_3['Severity'].iloc[-1],
        'Severity_Lag2': last_3['Severity'].iloc[-2],
        'Severity_Lag3': last_3['Severity'].iloc[-3],
        'Severity_RollMean3': last_3['Severity'].mean(),
    }])[features_lgbm] # Ensure correct column order
    
    # 2. Prepare GLM Inputs
    input_glm = pd.DataFrame([{
        'Time_Index': next_time_index,
        'month_sin': feat_sin, 'month_cos': feat_cos
    }])
    
    # 3. Get Predictions from Both
    pred_freq_lgbm = np.expm1(model_freq_lgbm.predict(input_lgbm)[0])
    pred_sev_lgbm = np.expm1(model_sev_lgbm.predict(input_lgbm)[0])
    
    pred_freq_glm = model_freq_glm.predict(input_glm)[0]
    pred_sev_glm = model_sev_glm.predict(input_glm)[0]
    
    # 4. ENSEMBLE: Average the predictions (50/50 split)
    # You can tweak this weight. Example: 0.6 * LGBM + 0.4 * GLM
    final_pred_freq = (pred_freq_lgbm + pred_freq_glm) / 2
    final_pred_sev = (pred_sev_lgbm + pred_sev_glm) / 2
    final_total_claim = final_pred_freq * final_pred_sev
    
    # Store Prediction
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': final_pred_freq,
            'Severity': final_pred_sev,
            'Total_Claim': final_total_claim
        })
        
    # Update History with the HYBRID prediction so next month's lags are stable
    new_row = pd.DataFrame([{
        'Year': period.year, 'Month': period.month,
        'Frequency': final_pred_freq, 'Severity': final_pred_sev
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)
    next_time_index += 1

# --- 4. FORMAT FOR SUBMISSION ---
results_df = pd.DataFrame(predictions)
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_hybrid.csv', index=False)

print("\nForecast Results (Hybrid Model):")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nSuccessfully saved to 'submission_hybrid.csv'")

Training LightGBM Models...
Training GLM Models...
Training Complete. Proceeding to Hybrid Forecast.

Forecast Results (Hybrid Model):
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 234.116896 52356576.998319 12257559303.864676
1      2025-09 241.576701 50485235.695374 12196056664.197708
2      2025-10 240.866853 51896230.583004 12500081731.022455
3      2025-11 234.669598 53127141.829978 12467325035.936821
4      2025-12 229.933321 51869963.970526 11926633085.759604

Successfully saved to 'submission_hybrid.csv'


Rich DF LightGBM

In [69]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# --- 1. EXTRACT RICH FEATURES FROM RAW DATA ---
# Ensure dates are datetime objects
claims_df['Tanggal Pasien Masuk RS'] = pd.to_datetime(claims_df['Tanggal Pasien Masuk RS'])
claims_df['Tanggal Pasien Keluar RS'] = pd.to_datetime(claims_df['Tanggal Pasien Keluar RS'])

# A. Calculate Length of Stay (LOS) in days. 
# If they leave the same day, we count it as 1 day to avoid zeros skewing averages.
claims_df['LOS'] = (claims_df['Tanggal Pasien Keluar RS'] - claims_df['Tanggal Pasien Masuk RS']).dt.days
claims_df['LOS'] = claims_df['LOS'].replace(0, 1)

# B. Convert Cashless indicator to binary
claims_df['Is_Cashless'] = (claims_df['Reimburse/Cashless'] == 'C').astype(int)

# C. Calculate Approval Ratio (How much of the bill we actually paid)
# Add a tiny number to denominator to prevent division by zero
claims_df['Approval_Ratio'] = claims_df['Nominal Klaim Yang Disetujui'] / (claims_df['Nominal Biaya RS Yang Terjadi'] + 1e-6)

# --- 2. AGGREGATE INTO A NEW 'RICH' MONTHLY DATAFRAME ---
monthly_data = []
grouped = claims_df.groupby(['Year', 'Month', 'Period'])

for (year, month, period), group in grouped:
    monthly_data.append({
        'Period': period,
        'Year': year,
        'Month': month,
        
        # Original Targets
        'Frequency': len(group),
        'Severity': group['Nominal Klaim Yang Disetujui'].mean(),
        'Total_Claim': group['Nominal Klaim Yang Disetujui'].sum(),
        
        # NEW: Portfolio Characteristics
        'Mean_LOS': group['LOS'].mean(),
        'Cashless_Ratio': group['Is_Cashless'].mean(),
        'Inpatient_Ratio': group['Is_IP'].mean(),
        'Mean_Approval_Ratio': group['Approval_Ratio'].mean(),
        'Unique_Claimants': group['Nomor Polis'].nunique()
    })

rich_df = pd.DataFrame(monthly_data).sort_values(by=['Year', 'Month']).reset_index(drop=True)

# --- 3. TIME-SERIES FEATURE ENGINEERING ---
rich_df['month_sin'] = np.sin(2 * np.pi * rich_df['Month']/12)
rich_df['month_cos'] = np.cos(2 * np.pi * rich_df['Month']/12)

# Create Lags and Rolling Means for Targets
for col in ['Frequency', 'Severity']:
    rich_df[f'{col}_Lag1'] = rich_df[col].shift(1)
    rich_df[f'{col}_Lag2'] = rich_df[col].shift(2)
    rich_df[f'{col}_Lag3'] = rich_df[col].shift(3)
    rich_df[f'{col}_RollMean3'] = rich_df[col].rolling(window=3).mean()

# Create Lags for the new Portfolio Characteristics
for col in ['Mean_LOS', 'Cashless_Ratio', 'Inpatient_Ratio', 'Mean_Approval_Ratio']:
    rich_df[f'{col}_Lag1'] = rich_df[col].shift(1)
    rich_df[f'{col}_RollMean3'] = rich_df[col].rolling(window=3).mean()

# Drop NaNs for training
train_df = rich_df.dropna().copy()

In [71]:
train_df.head()

,Period,Year,Month,Frequency,Severity,Total_Claim,Mean_LOS,Cashless_Ratio,Inpatient_Ratio,Mean_Approval_Ratio,...,Severity_Lag3,Severity_RollMean3,Mean_LOS_Lag1,Mean_LOS_RollMean3,Cashless_Ratio_Lag1,Cashless_Ratio_RollMean3,Inpatient_Ratio_Lag1,Inpatient_Ratio_RollMean3,Mean_Approval_Ratio_Lag1,Mean_Approval_Ratio_RollMean3
3,2024-04,2024,4,239,47870555.558787,11441062778.549999,1.870293,0.364017,0.669456,0.883684,...,67089342.682759,55327604.026867,2.118705,2.071653,0.377698,0.365828,0.705036,0.682523,0.891701,0.894892
4,2024-05,2024,5,263,46431413.766426,12211461820.570000,1.840304,0.376426,0.619772,0.994090,...,66632910.865882,48593771.660382,1.870293,1.943101,0.364017,0.372713,0.669456,0.664755,0.883684,0.923158
5,2024-06,2024,6,225,53889631.911244,12125167180.029999,2.035556,0.426667,0.471111,0.891448,...,51479345.655933,49397200.412152,1.840304,1.915384,0.376426,0.389036,0.619772,0.586780,0.994090,0.923074
6,2024-07,2024,7,257,58251037.858949,14970516729.750000,1.661479,0.439689,0.435798,125657076460.025284,...,47870555.558787,52857361.178873,2.035556,1.845779,0.426667,0.414260,0.471111,0.508894,0.891448,41885692153.970276
7,2024-08,2024,8,228,59267262.543465,13512935859.910000,2.210526,0.451754,0.416667,0.889106,...,46431413.766426,57135977.437886,1.661479,1.969187,0.439689,0.439370,0.435798,0.441192,125657076460.025284,41885692153.935280


In [72]:
# --- 4. TRAIN LIGHTGBM ---
features = [c for c in train_df.columns if 'Lag' in c or 'Roll' in c or 'month_' in c]

# Log transform targets
y_freq_log = np.log1p(train_df['Frequency'])
y_sev_log = np.log1p(train_df['Severity'])

X = train_df[features]

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 400,
    'learning_rate': 0.03, # Slightly slower learning rate for richer features
    'num_leaves': 25,
    'min_child_samples': 4,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbose': -1
}

print("Training Rich LightGBM for Frequency...")
model_freq = lgb.LGBMRegressor(**params).fit(X, y_freq_log)

print("Training Rich LightGBM for Severity...")
model_sev = lgb.LGBMRegressor(**params).fit(X, y_sev_log)

Training Rich LightGBM for Frequency...
Training Rich LightGBM for Severity...


In [73]:
# --- 5. RECURSIVE FORECASTING ---
last_date = pd.Period(f"{rich_df['Year'].iloc[-1]}-{rich_df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

current_history = rich_df.copy()
predictions = []

# Calculate the stable portfolio assumptions (Last known 3-month average)
stable_assumptions = {
    'Mean_LOS': current_history['Mean_LOS'].tail(3).mean(),
    'Cashless_Ratio': current_history['Cashless_Ratio'].tail(3).mean(),
    'Inpatient_Ratio': current_history['Inpatient_Ratio'].tail(3).mean(),
    'Mean_Approval_Ratio': current_history['Mean_Approval_Ratio'].tail(3).mean()
}

for period in forecast_range:
    p_month = period.month
    last_3 = current_history.tail(3)
    
    # Build Input Row
    input_dict = {
        'month_sin': np.sin(2 * np.pi * p_month/12),
        'month_cos': np.cos(2 * np.pi * p_month/12),
        
        # Target Features
        'Frequency_Lag1': last_3['Frequency'].iloc[-1],
        'Frequency_Lag2': last_3['Frequency'].iloc[-2],
        'Frequency_Lag3': last_3['Frequency'].iloc[-3],
        'Frequency_RollMean3': last_3['Frequency'].mean(),
        
        'Severity_Lag1': last_3['Severity'].iloc[-1],
        'Severity_Lag2': last_3['Severity'].iloc[-2],
        'Severity_Lag3': last_3['Severity'].iloc[-3],
        'Severity_RollMean3': last_3['Severity'].mean(),
        
        # Portfolio Features (Using historical data or stable assumptions if projecting)
        'Mean_LOS_Lag1': last_3['Mean_LOS'].iloc[-1] if not pd.isna(last_3['Mean_LOS'].iloc[-1]) else stable_assumptions['Mean_LOS'],
        'Mean_LOS_RollMean3': last_3['Mean_LOS'].mean() if len(last_3.dropna(subset=['Mean_LOS'])) > 0 else stable_assumptions['Mean_LOS'],
        
        'Cashless_Ratio_Lag1': last_3['Cashless_Ratio'].iloc[-1] if not pd.isna(last_3['Cashless_Ratio'].iloc[-1]) else stable_assumptions['Cashless_Ratio'],
        'Cashless_Ratio_RollMean3': last_3['Cashless_Ratio'].mean() if len(last_3.dropna(subset=['Cashless_Ratio'])) > 0 else stable_assumptions['Cashless_Ratio'],
        
        'Inpatient_Ratio_Lag1': last_3['Inpatient_Ratio'].iloc[-1] if not pd.isna(last_3['Inpatient_Ratio'].iloc[-1]) else stable_assumptions['Inpatient_Ratio'],
        'Inpatient_Ratio_RollMean3': last_3['Inpatient_Ratio'].mean() if len(last_3.dropna(subset=['Inpatient_Ratio'])) > 0 else stable_assumptions['Inpatient_Ratio'],
        
        'Mean_Approval_Ratio_Lag1': last_3['Mean_Approval_Ratio'].iloc[-1] if not pd.isna(last_3['Mean_Approval_Ratio'].iloc[-1]) else stable_assumptions['Mean_Approval_Ratio'],
        'Mean_Approval_Ratio_RollMean3': last_3['Mean_Approval_Ratio'].mean() if len(last_3.dropna(subset=['Mean_Approval_Ratio'])) > 0 else stable_assumptions['Mean_Approval_Ratio'],
    }
    
    # Ensure correct column order
    input_row = pd.DataFrame([input_dict])[features]
    
    # Predict
    pred_freq = np.expm1(model_freq.predict(input_row)[0])
    pred_sev = np.expm1(model_sev.predict(input_row)[0])
    
    pred_total_claim = pred_freq * pred_sev
    
    # Store Prediction
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total_claim
        })
        
    # Update History for next step
    new_row = pd.DataFrame([{
        'Year': period.year,
        'Month': period.month,
        'Frequency': pred_freq,
        'Severity': pred_sev,
        # Carry forward the assumptions so next month's lags work
        'Mean_LOS': stable_assumptions['Mean_LOS'],
        'Cashless_Ratio': stable_assumptions['Cashless_Ratio'],
        'Inpatient_Ratio': stable_assumptions['Inpatient_Ratio'],
        'Mean_Approval_Ratio': stable_assumptions['Mean_Approval_Ratio']
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)

In [74]:
# --- 6. FORMAT FOR SUBMISSION ---
results_df = pd.DataFrame(predictions)
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_rich_lgbm.csv', index=False)

print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nSuccessfully saved to 'submission_rich_lgbm.csv'")


Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 259.831892 54033484.146666 14039622392.962523
1      2025-09 258.907639 50929851.847287 13186127696.116346
2      2025-10 249.913130 51772406.872459 12938604244.242289
3      2025-11 247.993582 51576530.740116 12790648625.005024
4      2025-12 234.486649 48739207.708177 11428693486.747559

Successfully saved to 'submission_rich_lgbm.csv'


Direct Ridge

In [75]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge

# --- 1. SETUP & ISOLATED FEATURE ENGINEERING ---
df = final_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# Seasonality
df['month_sin'] = np.sin(2 * np.pi * df['Month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['Month']/12)

# We are now tracking Total Klaim directly as well
targets = ['Frequency', 'Severity', 'Total Klaim']

for col in targets:
    df[f'{col}_Lag1'] = df[col].shift(1)
    df[f'{col}_Lag2'] = df[col].shift(2)
    df[f'{col}_Lag3'] = df[col].shift(3)
    df[f'{col}_RollMean3'] = df[col].rolling(window=3).mean()

train_df = df.dropna().copy()

# --- 2. TRAIN INDEPENDENT RIDGE MODELS ---
# Ridge is highly regularized, preventing the wild swings of Tree models
alpha_val = 1.0 

# Model 1: Frequency (Only sees time and its own history)
features_freq = ['month_sin', 'month_cos', 'Frequency_Lag1', 'Frequency_Lag2', 'Frequency_Lag3', 'Frequency_RollMean3']
model_freq = Ridge(alpha=alpha_val)
model_freq.fit(train_df[features_freq], np.log1p(train_df['Frequency']))

# Model 2: Severity (Only sees time and its own history)
features_sev = ['month_sin', 'month_cos', 'Severity_Lag1', 'Severity_Lag2', 'Severity_Lag3', 'Severity_RollMean3']
model_sev = Ridge(alpha=alpha_val)
model_sev.fit(train_df[features_sev], np.log1p(train_df['Severity']))

# Model 3: Total Claim (DIRECT PREDICTION)
features_total = ['month_sin', 'month_cos', 'Total Klaim_Lag1', 'Total Klaim_Lag2', 'Total Klaim_Lag3', 'Total Klaim_RollMean3']
model_total = Ridge(alpha=alpha_val)
model_total.fit(train_df[features_total], np.log1p(train_df['Total Klaim']))

print("Independent Ridge Models Trained.")

# --- 3. RECURSIVE FORECASTING ---
last_date = pd.Period(f"{df['Year'].iloc[-1]}-{df['Month'].iloc[-1]}", freq='M')
target_start = pd.Period('2025-08', freq='M')
target_end = pd.Period('2025-12', freq='M')
forecast_range = pd.period_range(start=last_date + 1, end=target_end, freq='M')

current_history = df.copy()
predictions = []

for period in forecast_range:
    p_month = period.month
    feat_sin = np.sin(2 * np.pi * p_month/12)
    feat_cos = np.cos(2 * np.pi * p_month/12)
    last_3 = current_history.tail(3)
    
    # Predict Frequency
    row_freq = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                              'Frequency_Lag1': last_3['Frequency'].iloc[-1], 'Frequency_Lag2': last_3['Frequency'].iloc[-2],
                              'Frequency_Lag3': last_3['Frequency'].iloc[-3], 'Frequency_RollMean3': last_3['Frequency'].mean()}])[features_freq]
    pred_freq = np.expm1(model_freq.predict(row_freq)[0])
    
    # Predict Severity
    row_sev = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                             'Severity_Lag1': last_3['Severity'].iloc[-1], 'Severity_Lag2': last_3['Severity'].iloc[-2],
                             'Severity_Lag3': last_3['Severity'].iloc[-3], 'Severity_RollMean3': last_3['Severity'].mean()}])[features_sev]
    pred_sev = np.expm1(model_sev.predict(row_sev)[0])
    
    # Predict Total Claim DIRECTLY
    row_total = pd.DataFrame([{'month_sin': feat_sin, 'month_cos': feat_cos,
                               'Total Klaim_Lag1': last_3['Total Klaim'].iloc[-1], 'Total Klaim_Lag2': last_3['Total Klaim'].iloc[-2],
                               'Total Klaim_Lag3': last_3['Total Klaim'].iloc[-3], 'Total Klaim_RollMean3': last_3['Total Klaim'].mean()}])[features_total]
    pred_total = np.expm1(model_total.predict(row_total)[0])
    
    # Store
    if period >= target_start:
        predictions.append({
            'Month_Period': str(period),
            'id_month': str(period).replace('-', '_'),
            'Frequency': pred_freq,
            'Severity': pred_sev,
            'Total_Claim': pred_total # Using the direct prediction!
        })
        
    # Update History
    new_row = pd.DataFrame([{
        'Year': period.year, 'Month': period.month,
        'Frequency': pred_freq, 'Severity': pred_sev, 'Total Klaim': pred_total
    }])
    current_history = pd.concat([current_history, new_row], ignore_index=True)

# --- 4. FORMAT AND EXPORT ---
results_df = pd.DataFrame(predictions)
formatted_data = []

for index, row in results_df.iterrows():
    m_id = row['id_month']
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_ridge_direct.csv', index=False)

print("\nForecast Results:")
print(results_df[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nSaved to 'submission_ridge_direct.csv'")

Independent Ridge Models Trained.

Forecast Results:
  Month_Period  Frequency        Severity        Total_Claim
0      2025-08 238.082479 51179458.198422 12305576353.974308
1      2025-09 232.661628 56822963.469276 13337328304.152416
2      2025-10 263.894477 51706971.310517 13666087566.551418
3      2025-11 236.897665 50752414.656465 12214300466.838234
4      2025-12 231.424349 55955939.290475 13076111098.759466

Saved to 'submission_ridge_direct.csv'


c:\Users\Robben's Laptop\.conda\envs\data_science\Lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 6.264436495765169e-20.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
